# Lesson 6 : Memory and personalization (Context Provider / Memory Provider)

By using memory, you can keep the context in agent's conversation. For example, the assistant may keep your food's preference from past conversations, and it might recommend you the restaurant you favor when planning a trip later.

Context provider in Agent Framework handles context by using memory (for both short-term ephemeral memory and long-term persisted memory).

In this exercise, we explore 2 context providers - **Redis context provider** in Agent Framework and **Foundry memory provider** in Microsoft Foundry.

In Agent Framework, you can use built-in context providers - Azure AI search, Mem0, or Redis as backend. You can also configure your own custom provider.  
Redis context provider is one of these built-in providers, a provider with Redis backend.  
It's worth noting that **currently RediSearch doesn't support Japanese full text search**. Do not use Japanese prompt and instruction in this Redis context provider's example. (Use another provider for Japanese language.)

> Note : Here I don't go so far, but you can also configure Mem0 with Azure AI search or Redis backend as a vector store. (Today Mem0 is very popular for implementing memory.)

Microsoft Foundry also provides native memory store, which is based on Azure AI Search. In the latter part of this exercise, we use this memory store through foundry memory provider in Agent Framework.  

> Note : Please see [here](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/memory-usage?tabs=python) for directly using a memory store through Azure AI Projects SDK (```azure.ai.projects```).

---

Important Note : Currently, there is a bug in ```AzureAIClient``` that causes a warning when :
- You set ```use_latest_version=True``` option and get an agent.
- Run agent with the same ```AzureAIClient```, 2 or more times.

Please ignore warning.

---

## Redis context provider

### 1. Prepare Redis cache

Before writing code, we should provision Redis - by OSS installation or managed service.  
In this example, we use Azure Managed Redis, which is based on Redis Enterprise.

Open [Azure Portal](https://portal.azure.com) and create a new "Azure Managed Redis" resource with the following settings.

- Set "No Eviction" as "Eviction Policy".
- Select "Enterprise" as "Clustering Policy".
- Select "RediSearch" in "Modules".
- Select "Enable public access from all networks" in "Network access".
- Enable "Access Keys Authentication".

After Redis instance is created, copy endpoint url and access key in the created resource on Azure Portal.

### 2. Run agent with context provider

Now let's build an agent to use Redis context provider.

First we initilize the client object as usual.

In [1]:
from dotenv import load_dotenv
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we create an agent with Redis context provider as follows.  
Before running code, set Redis endpoint (which is retreived above) in ```REDIS_HOST```, and set access key (which is also retreived above) in ```REDIS_PASSWORD```.

> Note : In this example, we write settings directly into the code for demo purpose. (For security reasons, set these settings in environment variable and use it in production.)

**In this example, I have simply configured text-only retrieval for demo purpose**. (This example won't then perform semantic retrieval.)  
In production, please configure hybrid (text + vector) retrieval.

In [2]:
from agent_framework import Agent
from agent_framework.redis import RedisContextProvider

# ToDo : fill your settings
REDIS_HOST = "xxxxx.xxxxx.redis.azure.net:10000"
REDIS_PASSWORD = "xxxxxxxxxxxxxxxxxxxx"

redis_url = f"rediss://:{REDIS_PASSWORD}@{REDIS_HOST}"

provider1 = RedisContextProvider(
    redis_url=redis_url,
    source_id="redis_basic_chat",
    index_name="index01",
    application_id="app01",
    agent_id="agent01",
    user_id="demouser01",
)

agent1 = Agent(
    name="AgentWithContext01",
    client=client,
    instructions=(
        "You are a helpful assistant. "
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider1],
)

Now let's run agent with the following user message which tells your food preference.  
In the backend, Redis index is generated and the message context is stored in the index.

In [3]:
from IPython.display import Markdown, display

result = await agent1.run("My favorite food is sushi.")
display(Markdown(result.text))

I don’t have any stored context from earlier in this chat.

Noted: your favorite food is sushi.

Next let's ask my favor in this agent as follows.  
In the backend, the agent retrieves the context associated with this message, and this context is used to create a response. (As I have mentioned above, text-only retrieval is applied in this example. So direct question (primitive question) is issued in this example.)

In [4]:
result = await agent1.run("What is my favorite food?")
display(Markdown(result.text))

Your favorite food is sushi.

You can scope the context by using ```application_id```, ```agent_id```, ```user_id``` and ```thread_id``` in ```RedisContextProvider``` configuration.

Now I create a new agent which has context provider with different user id "demouser02".  
As you see, the agent doesn't know my preference, because it's a context of different user.

In [5]:
provider2 = RedisContextProvider(
    redis_url=redis_url,
    source_id="redis_basic_chat",
    index_name="index01",
    application_id="app01",
    agent_id="agent01",
    user_id="demouser02",
)

agent2 = Agent(
    name="AgentWithContext02",
    client=client,
    instructions=(
        "You are a helpful assistant."
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider2],
)

result = await agent2.run("What is my favorite food?")
display(Markdown(result.text))

I don’t have any stored context about your favorite food, so I can’t know it yet. Tell me what it is (or give me a few candidates), and I’ll remember it for this chat.

Now we go back again to the user "demouser01", and ask the preference.  
The agent object is newly created (therefore, started a new thread), but it remembers my preference, because it's stored in the context provider.

In [6]:
client = AzureAIClient(
    credential=credential,
    agent_name="AgentWithContext01",
    agent_version="1",
)

agent1 = Agent(
    client=client,
    instructions=(
        "You are a helpful assistant."
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider1],
)

result = await agent1.run("What is my favorite food?")
display(Markdown(result.text))

Your favorite food is sushi.

If you want to see the stored messages in Redis index for debugging, you can run as follows.

In [7]:
all_data = await provider1.search_all()
for item in all_data:
    print(item)

{'id': 'context:01KF008VBT6TDZMS1GFH9GKRTR', 'role': 'user', 'content': 'My favorite food is sushi.', 'conversation_id': 'resp_0c2d274c219e471f00696873d0807081909b0b018ad6f1e65c', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01'}
{'id': 'context:01KF008VBX93Y1GCEV1Y6JT7MP', 'role': 'assistant', 'content': 'I don’t have any stored context from earlier in this chat.\n\nNoted: your favorite food is sushi.', 'conversation_id': 'resp_0c2d274c219e471f00696873d0807081909b0b018ad6f1e65c', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01', 'author_name': 'AgentWithContext01'}
{'id': 'context:01KF0097JSCTE5SR75V61CRSN4', 'role': 'user', 'content': 'What is my favorite food?', 'conversation_id': 'resp_08441b9242cb055f00696873ddac348197a04573ea30aa9e58', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01'}
{'id': 'context:01KF0097JSCTE5SR75V61CRSN5', 'role': 'assistant', 'content': 'Your favorite food is sushi.', 'conversation_

### 3. Clean-up

Remove index as follows, when you don't use context provider any more.

In [8]:
await provider1.redis_index.delete()

## Foundry Memory Provider

In the next example, we explore foundry memory provider, which uses a memory store in Microsoft Foundry.

### 1. Prerequisites

Currently (Mar 2026), in order to run the following example, your foundry project's managed identity should have Azure AI User role on its parent foundry resource.  
Before running this example, assign this role as follows.

- Go to foundry resource (not foundry project resource) in Azure Portal.
- Select "Access Control (IAM)".
- Assign "Azure AI User" role to foundry project resource (not parent resource).

### 2. Deploy an embedding model

Before using a foundry memory provider in Agent Framework, we should provision a chat model, an embedding model, and a memory store in Microsoft Foundry.

We have already deployed a chat model in [Readme.md](./Readme.md), and please additionally deploy an embedding model (such as, ```text-embedding-3-small```) in Microsoft Foundry.  
After you have deployed, please fill the following.

In [9]:
# ToDo : fill deployment name
embedding_model="[Fill-Your-Embedding-Model-Name]"

### 3. Create a memory store in Foundry

Next we create a memory store in Microsoft Foundry.  
You can create a memory store using Foundry Portal UI, but in this example, we create a memory store using Azure AI Projects SDK (```azure.ai.projects```) as follows.

> Note : Azure AI Projects SDK (```azure.ai.projects```) is a library to build and operate components in Microsoft Foundry.  
> As I have mentioned in Lesson 1, ```AzureAIClient``` in Agent Framework is built using this library, and Agent Framework also installs Azure AI Projects SDK as a dependant package.  
> ```AzureAIClient``` doesn't provide a capability to create a Foundry memory store, and you should directly use Azure AI Projects SDK in order to create a memory store.

In [10]:
import os
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import (
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
)

project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=credential,
)

memory_store_name = "my_memory_store"

options = MemoryStoreDefaultOptions(
    chat_summary_enabled=False,
    user_profile_enabled=True,
    user_profile_details="Avoid irrelevant or sensitive data, such as age, financials, precise location, and credentials"
)

definition = MemoryStoreDefaultDefinition(
    chat_model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    embedding_model=embedding_model,
    options=options
)

memory_store = await project_client.beta.memory_stores.create(
    name=memory_store_name,
    definition=definition,
    description="Memory store for customer support agent",
)

print(f"Created memory store: {memory_store.name}")

Created memory store: my_memory_store


### 4. Run agent with Foundry memory provider

Now let's build an agent to use Foundry memory provider as follows.

In the following ```FoundryMemoryProvider```, the ```scope``` property specifies a scope to a specific user. If it's not set, the session (see [Lesson3](./03_session.ipynb)) will be used as scope, which means memories are only shared within the same session.  
The ```update_delay``` property is used to set a delay after each interaction for batch updates to reduce costs. In this example, we set zero (no wait)
for demo purpose.

In [11]:
from agent_framework.azure import FoundryMemoryProvider

foundry_memory_provider = FoundryMemoryProvider(
    project_client=project_client,
    memory_store_name=memory_store_name,
    scope="demouser01",
    update_delay=0,
)

agent3 = Agent(
    name="AgentWithFoundryMemory",
    client=client,
    instructions=(
        "You are a helpful assistant. "
        "The memories from previous interactions are automatically provided to you."
    ),
    tools=[],
    context_providers=[foundry_memory_provider],
)

Let's run agent with the following user message which tells your food preference.  
In the backend, the conversation context is saved in memory store by running ```project_client.beta.memory_stores.begin_update_memories()``` in Azure AI Projects SDK. (Text is embedded and saved in store.)

> Note : You can view the stored contents in memory store by running ```project_client.beta.memory_stores.search_memories()``` in Azure AI Projects SDK.

In [12]:
import asyncio

result = await agent3.run("My favorite food is sushi.")
display(Markdown(result.text))

# wait until memory is updated
# (because begin_update_memories() is asynchronous call and AF doesn't wait the result.)
await asyncio.sleep(8)

Got it — I’ll remember that your favorite food is sushi.

Let's ask my favor in this agent as follows.  
In the backend, the agent retrieves the context associated with this message, and this context is used to create a response.

In [13]:
result = await agent3.run("What is my favorite food?")
display(Markdown(result.text))

Your favorite food is **sushi**.

In [14]:
result = await agent3.run("What do you remember about my preferences?")
display(Markdown(result.text))

I remember that your favorite food is sushi.

### 5. Clean-up

Remove memory store as follows, when you don't need Foundry memory any more.

In [15]:
await project_client.beta.memory_stores.delete("my_memory_store");